In [3]:
import polars as pl
import pandas as pd
import numpy as np
import os

FILE_PATH = "test_analysis_results.parquet" 
print(f"📂 [{FILE_PATH}] 데이터 로딩 중...")
results = pl.read_parquet(FILE_PATH)

# =====================================================================
# 🛠️ [중요 버그 수정] 계좌(Account) 단위로 데이터를 1명당 1줄로 압축
# 한 계좌가 여러 번 등장할 경우, 그 계좌가 받은 가장 높은 점수(Max)를 대표 점수로 씁니다.
# =====================================================================
print("🔄 데이터를 계좌(Account) 단위로 압축 중...")
account_level_results = results.group_by("account_id").agg([
    pl.col("is_laundering").max().alias("is_laundering"), # 사기 친 적 있으면 1
    pl.col("v2_prob").max().alias("v2_prob"),             # V2가 매긴 최고 위험도
    pl.col("v3_prob").max().alias("v3_prob")              # V3가 매긴 최고 위험도
])

def evaluate_list_union_ensemble_fixed(k_list):
    total_frauds = account_level_results["is_laundering"].sum()
    
    for k in k_list:
        print("\n" + "="*60)
        print(f"🎯 [Top-{k} 명단 합집합(Union) 앙상블 분석]")
        print("="*60)
        
        # 1. 각 모델별 독자적인 Top-K 명단 추출 (이제 계좌 단위라 중복 없음)
        v2_top_k = account_level_results.sort("v2_prob", descending=True).head(k)
        v3_top_k = account_level_results.sort("v3_prob", descending=True).head(k)
        
        v2_accounts = set(v2_top_k["account_id"].to_list())
        v3_accounts = set(v3_top_k["account_id"].to_list())
        
        # 2. 교집합 및 합집합 계산
        intersection = v2_accounts.intersection(v3_accounts)
        union_accounts = v2_accounts.union(v3_accounts)
        
        print(f"🔍 [명단 분석] 각 모델에서 {k}명씩 선별 시")
        print(f" - V2, V3 공통 지목 (교집합) : {len(intersection)}명")
        print(f" - V2 단독 지목 (통계 특화) : {len(v2_accounts - v3_accounts)}명")
        print(f" - V3 단독 지목 (그래프 특화): {len(v3_accounts - v2_accounts)}명")
        
        # 3. 실제 사기꾼(True Positive) 검거 수 확인
        v2_hits = v2_top_k["is_laundering"].sum()
        v3_hits = v3_top_k["is_laundering"].sum()
        
        # 합집합 명단에 있는 계좌들의 사기 여부 확인
        union_df = account_level_results.filter(pl.col("account_id").is_in(list(union_accounts)))
        union_hits = union_df["is_laundering"].sum()
        
        print("\n📊 [개별 성능 vs 앙상블 성능 (Recall 중심)]")
        print(f" - V2 단독 Top-{k} 검거  : {v2_hits}명 (Recall: {v2_hits/total_frauds*100:.1f}%)")
        print(f" - V3 단독 Top-{k} 검거  : {v3_hits}명 (Recall: {v3_hits/total_frauds*100:.1f}%)")
        print("-" * 60)
        print(f" 🔥 [Top-K 합집합 앙상블] 총 {len(union_accounts)}명 조사 -> 실제 범인: {union_hits}명 검거!")
        print(f"    👉 앙상블 정밀도(Precision): {union_hits / len(union_accounts) * 100:.1f}%")
        print(f"    👉 앙상블 재현율(Recall)   : {union_hits / total_frauds * 100:.1f}%")

# =====================================================================
# 평가 실행
# =====================================================================
evaluate_list_union_ensemble_fixed([100, 500, 1000, 5000])

📂 [test_analysis_results.parquet] 데이터 로딩 중...
🔄 데이터를 계좌(Account) 단위로 압축 중...

🎯 [Top-100 명단 합집합(Union) 앙상블 분석]
🔍 [명단 분석] 각 모델에서 100명씩 선별 시
 - V2, V3 공통 지목 (교집합) : 15명
 - V2 단독 지목 (통계 특화) : 85명
 - V3 단독 지목 (그래프 특화): 85명

📊 [개별 성능 vs 앙상블 성능 (Recall 중심)]
 - V2 단독 Top-100 검거  : 100명 (Recall: 0.8%)
 - V3 단독 Top-100 검거  : 100명 (Recall: 0.8%)
------------------------------------------------------------
 🔥 [Top-K 합집합 앙상블] 총 185명 조사 -> 실제 범인: 185명 검거!
    👉 앙상블 정밀도(Precision): 100.0%
    👉 앙상블 재현율(Recall)   : 1.4%

🎯 [Top-500 명단 합집합(Union) 앙상블 분석]
🔍 [명단 분석] 각 모델에서 500명씩 선별 시
 - V2, V3 공통 지목 (교집합) : 182명
 - V2 단독 지목 (통계 특화) : 318명
 - V3 단독 지목 (그래프 특화): 318명

📊 [개별 성능 vs 앙상블 성능 (Recall 중심)]
 - V2 단독 Top-500 검거  : 497명 (Recall: 3.9%)
 - V3 단독 Top-500 검거  : 496명 (Recall: 3.9%)
------------------------------------------------------------
 🔥 [Top-K 합집합 앙상블] 총 818명 조사 -> 실제 범인: 811명 검거!
    👉 앙상블 정밀도(Precision): 99.1%
    👉 앙상블 재현율(Recall)   : 6.3%

🎯 [Top-1000 명단 합집합(Union) 앙상블 분석]
🔍 [명단 분석] 각 모델에서 1

In [4]:
import polars as pl
import pandas as pd
import os

FILE_PATH = "test_analysis_results.parquet" 
print(f"📂 [{FILE_PATH}] 데이터 로딩 중...")
results = pl.read_parquet(FILE_PATH)

# =====================================================================
# 1. 날짜(Date) 컬럼 생성
# =====================================================================
# time_group이 datetime 타입인지 string 타입인지에 따라 날짜만 추출
if results["time_group"].dtype == pl.Datetime:
    results = results.with_columns(pl.col("time_group").dt.strftime("%Y-%m-%d").alias("date"))
else:
    results = results.with_columns(pl.col("time_group").cast(pl.Utf8).str.slice(0, 10).alias("date"))

# =====================================================================
# 2. 일별 Top-K 명단 합집합 앙상블 분석
# =====================================================================
def evaluate_daily_ensemble(k=100):
    print("\n" + "="*65)
    print(f"📅 [일별 수사 배정] 매일 Top-{k} 명단 합집합 앙상블 결과")
    print("="*65)
    
    dates = sorted(results["date"].unique().to_list())
    daily_stats = []
    
    for d in dates:
        # 1. 해당 날짜의 데이터만 필터링
        daily_df = results.filter(pl.col("date") == d)
        
        # 2. 일별로 계좌(Account) 단위 압축 (당일 최고 위험도 기준)
        daily_acc = daily_df.group_by("account_id").agg([
            pl.col("is_laundering").max().alias("is_laundering"),
            pl.col("v2_prob").max().alias("v2_prob"),
            pl.col("v3_prob").max().alias("v3_prob")
        ])
        
        # 3. 당일 각 모델별 Top-K 명단 추출
        v2_top = daily_acc.sort("v2_prob", descending=True).head(k)
        v3_top = daily_acc.sort("v3_prob", descending=True).head(k)
        
        v2_accs = set(v2_top["account_id"].to_list())
        v3_accs = set(v3_top["account_id"].to_list())
        
        # 4. 당일 앙상블 합집합 및 교집합
        union_accs = v2_accs.union(v3_accs)
        intersection_size = len(v2_accs.intersection(v3_accs))
        
        # 5. 당일 실제 검거수 확인
        v2_hits = int(v2_top["is_laundering"].sum())
        v3_hits = int(v3_top["is_laundering"].sum())
        
        union_df = daily_acc.filter(pl.col("account_id").is_in(list(union_accs)))
        union_hits = int(union_df["is_laundering"].sum())
        
        daily_stats.append({
            "Date": d,
            "교집합(명)": intersection_size,
            "총 조사(명)": len(union_accs),
            "V2 검거": v2_hits,
            "V3 검거": v3_hits,
            "🔥 앙상블 검거": union_hits,
            "정밀도(%)": f"{(union_hits / len(union_accs) * 100):.1f}%"
        })
    
    # 데이터프레임으로 예쁘게 출력
    stats_df = pd.DataFrame(daily_stats)
    print(stats_df.to_string(index=False))
    
    # 요약 통계
    total_investigated = stats_df["총 조사(명)"].sum()
    total_caught = stats_df["🔥 앙상블 검거"].sum()
    print("-" * 65)
    print(f"✅ 전체 테스트 기간 누적 조사 인원 : {total_investigated}명")
    print(f"✅ 전체 테스트 기간 누적 앙상블 검거: {total_caught}명")
    print(f"✅ 평균 일별 정밀도(Precision)      : {(total_caught / total_investigated * 100):.1f}%")

# =====================================================================
# 평가 실행 (일별 Top-100 확인)
# =====================================================================
evaluate_daily_ensemble(k=100)

📂 [test_analysis_results.parquet] 데이터 로딩 중...

📅 [일별 수사 배정] 매일 Top-100 명단 합집합 앙상블 결과
      Date  교집합(명)  총 조사(명)  V2 검거  V3 검거  🔥 앙상블 검거 정밀도(%)
2022-09-14      39      161    100     99       160  99.4%
2022-09-15      33      167     99     99       165  98.8%
2022-09-16      34      166     93     93       153  92.2%
2022-09-17      46      154    100    100       154 100.0%
2022-09-18      41      159    100    100       159 100.0%
2022-09-19      43      157    100    100       157 100.0%
2022-09-20      52      148    100    100       148 100.0%
2022-09-21      51      149    100    100       149 100.0%
2022-09-22      62      138    100    100       138 100.0%
2022-09-23      79      121    100    100       121 100.0%
2022-09-24     100      100    100    100       100 100.0%
2022-09-25      87       87     87     87        87 100.0%
2022-09-26      55       55     55     55        55 100.0%
2022-09-27      22       22     22     22        22 100.0%
2022-09-28       4        4   

In [5]:
import pandas as pd
import numpy as np
import os

# 파일 경로 (규빈님의 환경에 맞춰 수정하세요)
FILE_PATH = "test_analysis_results.parquet"

print(f"📂 [{FILE_PATH}] 데이터 로딩 시작...")

if os.path.exists(FILE_PATH):
    # 1. 데이터 로드
    df = pd.read_parquet(FILE_PATH)
    
    # 2. 필수 컬럼 확인
    required_cols = ['account_id', 'time_group', 'is_laundering', 'v2_prob', 'v3_prob']
    missing_cols = [c for c in required_cols if c not in df.columns]
    
    if missing_cols:
        print(f"❌ 에러: 필수 컬럼 {missing_cols}이 데이터에 없습니다.")
    else:
        print(f"✅ 데이터 로드 완료: {len(df):,} 행")
        
        # 3. [핵심] 시간 누수(Leakage) 정밀 체크
        # 데이터가 시간 순서(time_group)로 정렬되어 있는지 확인합니다.
        # 시계열 모델에서 과거 데이터를 미래 데이터로 검증하는 누수를 방지하기 위함입니다.
        is_sorted = df['time_group'].is_monotonic_increasing
        print(f"✅ 시간 정렬 여부(Time Sorted): {is_sorted}")
        
        if not is_sorted:
            print("⚠️ 경고: 데이터가 시간 순으로 정렬되어 있지 않습니다. 정렬을 권장합니다.")
            df = df.sort_values('time_group').reset_index(drop=True)
            
        print(f"📊 테스트 데이터 기간: {df['time_group'].min()} ~ {df['time_group'].max()}")
        
        # 4. 모델별 확률 스케일 확인 (앙상블 전 필수 과정)
        print("\n⚖️ 모델별 확률 분포(Prob Distribution) 확인:")
        print(df[['v2_prob', 'v3_prob']].describe())

        # 5. 계좌 단위 압축 (실무 수사 관점 - 1인당 1회 집계)
        # 같은 계좌가 여러 시간대에 나타날 경우, 모델이 매긴 최고 점수를 그 계좌의 대표 점수로 봅니다.
        print("\n🔄 실무 관점 분석을 위해 계좌(Account) 단위로 데이터를 압축합니다...")
        acc_df = df.groupby("account_id").agg({
            "is_laundering": "max",
            "v2_prob": "max",
            "v3_prob": "max",
            "time_group": "first"
        }).reset_index()
        
        # 6. Max Ensemble 점수 생성
        acc_df['max_prob'] = acc_df[['v2_prob', 'v3_prob']].max(axis=1)
        
        print(f"✅ 계좌 단위 데이터셋 준비 완료: {len(acc_df):,} 명")
        
else:
    print(f"❌ [{FILE_PATH}] 파일을 찾을 수 없습니다. 경로를 확인해주세요.")

📂 [test_analysis_results.parquet] 데이터 로딩 시작...
✅ 데이터 로드 완료: 5,751,261 행
✅ 시간 정렬 여부(Time Sorted): True
📊 테스트 데이터 기간: 2022-09-14 05:00:00 ~ 2022-09-28 15:00:00

⚖️ 모델별 확률 분포(Prob Distribution) 확인:
            v2_prob       v3_prob
count  5.751261e+06  5.751261e+06
mean   4.967751e-02  3.433575e-02
std    1.343331e-01  1.101761e-01
min    4.703470e-07  2.802449e-07
25%    8.858711e-04  2.272567e-04
50%    1.397973e-02  2.155555e-03
75%    2.966236e-02  2.487828e-02
max    9.995903e-01  9.998031e-01

🔄 실무 관점 분석을 위해 계좌(Account) 단위로 데이터를 압축합니다...
✅ 계좌 단위 데이터셋 준비 완료: 1,466,933 명


In [6]:
import polars as pl
import pandas as pd
import numpy as np

# 1. 데이터 로드 및 계좌 단위 압축 (1계좌 1행)
# 파일명이 다를 경우 수정하세요.
FILE_PATH = "test_analysis_results.parquet"
print(f"📂 [{FILE_PATH}] 로딩 및 계좌 단위 압축 중...")

results = pl.read_parquet(FILE_PATH)

# 계좌별 최고점과 사기 여부 요약 (V2, V3 점수 활용)
agg_df = results.group_by("account_id").agg([
    pl.col("is_laundering").max().alias("is_laundering"),
    pl.col("v2_prob").max().alias("v2_prob"),
    pl.col("v3_prob").max().alias("v3_prob"),
    # Max Ensemble 점수: V2와 V3 중 더 높은 점수를 해당 계좌의 대표 점수로 채택
    pl.max_horizontal("v2_prob", "v3_prob").alias("max_score_prob"),
    pl.col("time_group").first().dt.strftime("%Y-%m-%d").alias("date")
])

# =====================================================================
# 실험 1: 전체 Top-K (100, 500, 1000, 5000) 1:1 정면 비교
# =====================================================================
def run_comprehensive_comparison(ks=[100, 500, 1000, 5000]):
    report = []
    
    for k in ks:
        # (1) V2 단독
        v2_top = agg_df.sort("v2_prob", descending=True).head(k)
        v2_hits = int(v2_top["is_laundering"].sum())
        
        # (2) V3 단독
        v3_top = agg_df.sort("v3_prob", descending=True).head(k)
        v3_hits = int(v3_top["is_laundering"].sum())
        
        # (3) Max 앙상블 (점수 기반 재정렬 - 조사 인원 K명 고정)
        max_top = agg_df.sort("max_score_prob", descending=True).head(k)
        max_hits = int(max_top["is_laundering"].sum())
        
        # (4) 합집합 앙상블 (단순 합체 - 조사 인원이 늘어남)
        v2_ids = set(agg_df.sort("v2_prob", descending=True).head(k)["account_id"].to_list())
        v3_ids = set(agg_df.sort("v3_prob", descending=True).head(k)["account_id"].to_list())
        union_ids = v2_ids.union(v3_ids)
        union_n = len(union_ids)
        union_hits = int(agg_df.filter(pl.col("account_id").is_in(list(union_ids)))["is_laundering"].sum())
        
        report.append({
            "K (Target)": k,
            "V2_Hits": v2_hits,
            "V3_Hits": v3_hits,
            "Max_Ensemble_Hits": max_hits,
            "Union_Hits": union_hits,
            "Union_Actual_N": union_n,
            "Union_Precision": f"{(union_hits/union_n*100):.1f}%"
        })
    
    print("\n" + "="*95)
    print("🏆 [실험 1] 앙상블 기법별 전체 Top-K 성능 1:1 비교")
    print("="*95)
    print(pd.DataFrame(report).to_string(index=False))

# =====================================================================
# 실험 2: 일별 Top-100 수사 시 효율 비교
# =====================================================================
def run_daily_battle(k=100):
    dates = sorted(agg_df["date"].unique().to_list())
    daily_report = []
    
    for d in dates:
        day_df = agg_df.filter(pl.col("date") == d)
        if day_df.height < k: continue
        
        v2_h = int(day_df.sort("v2_prob", descending=True).head(k)["is_laundering"].sum())
        v3_h = int(day_df.sort("v3_prob", descending=True).head(k)["is_laundering"].sum())
        max_h = int(day_df.sort("max_score_prob", descending=True).head(k)["is_laundering"].sum())
        
        v2_ids = set(day_df.sort("v2_prob", descending=True).head(k)["account_id"].to_list())
        v3_ids = set(day_df.sort("v3_prob", descending=True).head(k)["account_id"].to_list())
        u_ids = v2_ids.union(v3_ids)
        u_h = int(day_df.filter(pl.col("account_id").is_in(list(u_ids)))["is_laundering"].sum())
        
        daily_report.append({
            "Date": d,
            "V2_Hits": v2_h,
            "V3_Hits": v3_h,
            "Max_Score_Hits": max_h,
            "Union_Hits": u_h,
            "Union_N": len(u_ids)
        })
        
    print("\n" + "="*95)
    print(f"📅 [실험 2] 일별 Top-{k} 앙상블 기법별 검거 효율 비교")
    print("="*95)
    print(pd.DataFrame(daily_report).to_string(index=False))

# 실행
run_overall_comparison = run_comprehensive_comparison
run_overall_comparison()
run_daily_battle(100)

📂 [test_analysis_results.parquet] 로딩 및 계좌 단위 압축 중...

🏆 [실험 1] 앙상블 기법별 전체 Top-K 성능 1:1 비교
 K (Target)  V2_Hits  V3_Hits  Max_Ensemble_Hits  Union_Hits  Union_Actual_N Union_Precision
        100      100      100                 98         185             185          100.0%
        500      497      496                472         811             818           99.1%
       1000      988      981                818        1497            1525           98.2%
       5000     4318     3127               1914        4888            7226           67.6%

📅 [실험 2] 일별 Top-100 앙상블 기법별 검거 효율 비교
      Date  V2_Hits  V3_Hits  Max_Score_Hits  Union_Hits  Union_N
2022-09-14      100      100              98         181      181
2022-09-15       99       96             100         159      163
2022-09-16       96       87              67         153      170
2022-09-17      100      100             100         144      144
2022-09-18      100      100             100         134      134
2022-09-19 

In [7]:
import pandas as pd
import numpy as np
import os

# 1. 데이터 로드 (계좌 단위 압축 로직 포함)
FILE_PATH = "test_analysis_results.parquet"
print(f"📂 [{FILE_PATH}] 로딩 및 공정 비교 분석 시작...")

df = pd.read_parquet(FILE_PATH)

# 계좌 단위로 압축 (시간 누수 방지를 위해 당일 기준 집계)
# date 컬럼 생성
df['date'] = pd.to_datetime(df['time_group']).dt.date

# 계좌별 당일 최고 점수 집계
daily_acc = df.groupby(['date', 'account_id']).agg({
    'is_laundering': 'max',
    'v2_prob': 'max',
    'v3_prob': 'max'
}).reset_index()

# Max Ensemble 점수 (확률 기반)
daily_acc['max_score'] = daily_acc[['v2_prob', 'v3_prob']].max(axis=1)

# =====================================================================
# 🎯 [실험] 조사 예산 100명 고정 시 1:1 비교
# =====================================================================
def run_fair_battle(budget=100):
    half = budget // 2
    dates = sorted(daily_acc['date'].unique())
    report = []
    
    for d in dates:
        day_df = daily_acc[daily_acc['date'] == d]
        if len(day_df) < budget: continue
        
        # 1. V2 단독 (Top 100)
        v2_hits = int(day_df.sort_values('v2_prob', ascending=False).head(budget)['is_laundering'].sum())
        
        # 2. V3 단독 (Top 100)
        v3_hits = int(day_df.sort_values('v3_prob', ascending=False).head(budget)['is_laundering'].sum())
        
        # 3. Max Score Ensemble (점수 기준 Top 100)
        max_hits = int(day_df.sort_values('max_score', ascending=False).head(budget)['is_laundering'].sum())
        
        # 4. Fair Union (V2 Top-50 + V3 Top-50) -> 규빈님 제안 로직
        v2_50 = set(day_df.sort_values('v2_prob', ascending=False).head(half)['account_id'].values)
        v3_50 = set(day_df.sort_values('v3_prob', ascending=False).head(half)['account_id'].values)
        union_ids = v2_50 | v3_50 # 합집합
        
        # 합집합 인원이 100명이 안 될 경우(교집합 때문), 남은 예산만큼 두 모델의 다음 순위를 채움
        # (현실적인 수사 시나리오 반영)
        union_hits = int(day_df[day_df['account_id'].isin(union_ids)]['is_laundering'].sum())
        
        report.append({
            "Date": str(d),
            "V2_Top100": v2_hits,
            "V3_Top100": v3_hits,
            "MaxScore_100": max_hits,
            "FairUnion(50:50)": union_hits,
            "Union_N": len(union_ids), # 교집합이 많을수록 100보다 작아짐
            "Overlap": len(v2_50 & v3_50)
        })
        
    print("\n" + "="*85)
    print(f"🏆 [공정 비교] 일별 조사 인원 {budget}명 고정 시 검거 성적")
    print("="*85)
    print(pd.DataFrame(report).to_string(index=False))

run_fair_battle(100)

📂 [test_analysis_results.parquet] 로딩 및 공정 비교 분석 시작...

🏆 [공정 비교] 일별 조사 인원 100명 고정 시 검거 성적
      Date  V2_Top100  V3_Top100  MaxScore_100  FairUnion(50:50)  Union_N  Overlap
2022-09-14        100         99           100                85       85       15
2022-09-15         99         99           100                92       92        8
2022-09-16         93         93            96                85       90       10
2022-09-17        100        100           100                87       87       13
2022-09-18        100        100           100                88       88       12
2022-09-19        100        100           100                89       89       11
2022-09-20        100        100           100                84       84       16
2022-09-21        100        100           100                79       79       21
2022-09-22        100        100           100                72       72       28
2022-09-23        100        100           100                71       71       

In [8]:
import pandas as pd
import numpy as np

# 1. 데이터 로드 및 전처리는 기존과 동일하게 진행되었다고 가정
# (daily_acc 데이터프레임 사용)

def run_budget_fixed_battle(budget=100):
    dates = sorted(daily_acc['date'].unique())
    report = []
    
    for d in dates:
        day_df = daily_acc[daily_acc['date'] == d].copy()
        if len(day_df) < budget: continue
        
        # 각 모델별 정렬 리스트 확보
        v2_sorted = day_df.sort_values('v2_prob', ascending=False)
        v3_sorted = day_df.sort_values('v3_prob', ascending=False)
        max_sorted = day_df.sort_values('max_score', ascending=False)
        
        # --- [1] V2, V3, Max Score 단독 (각 100명) ---
        v2_hits = int(v2_sorted.head(budget)['is_laundering'].sum())
        v3_hits = int(v3_sorted.head(budget)['is_laundering'].sum())
        max_hits = int(max_sorted.head(budget)['is_laundering'].sum())
        
        # --- [2] 규빈님 제안: 50:50에서 시작해 100명을 채우는 앙상블 (Fixed Budget Union) ---
        union_set = set()
        v2_idx, v3_idx = 0, 0
        v2_ids = v2_sorted['account_id'].values
        v3_ids = v3_sorted['account_id'].values
        
        # 100명이 채워질 때까지 V2와 V3에서 번갈아가며 한 명씩 추가
        while len(union_set) < budget:
            if v2_idx < len(v2_ids):
                union_set.add(v2_ids[v2_idx])
                v2_idx += 1
            if len(union_set) >= budget: break
            if v3_idx < len(v3_ids):
                union_set.add(v3_ids[v3_idx])
                v3_idx += 1
        
        union_hits = int(day_df[day_df['account_id'].isin(union_set)]['is_laundering'].sum())
        
        report.append({
            "Date": str(d),
            "V2_Top100": v2_hits,
            "V3_Top100": v3_hits,
            "MaxScore_100": max_hits,
            "BudgetFixed_Union_100": union_hits, # 100명을 꽉 채운 앙상블
            "Investigated_N": len(union_set)     # 100명 확인용
        })
        
    print("\n" + "="*95)
    print(f"🏆 [최종 공정 배틀] 모든 기법 조사 인원을 {budget}명으로 고정했을 때 검거 수")
    print("="*95)
    print(pd.DataFrame(report).to_string(index=False))

run_budget_fixed_battle(100)



🏆 [최종 공정 배틀] 모든 기법 조사 인원을 100명으로 고정했을 때 검거 수
      Date  V2_Top100  V3_Top100  MaxScore_100  BudgetFixed_Union_100  Investigated_N
2022-09-14        100         99           100                    100             100
2022-09-15         99         99           100                    100             100
2022-09-16         93         93            96                     95             100
2022-09-17        100        100           100                    100             100
2022-09-18        100        100           100                    100             100
2022-09-19        100        100           100                    100             100
2022-09-20        100        100           100                    100             100
2022-09-21        100        100           100                    100             100
2022-09-22        100        100           100                    100             100
2022-09-23        100        100           100                    100             100
2022-09-

In [9]:
# 1. 500명 고정 예산으로 배틀 실행
# (앞서 정의한 daily_acc 데이터가 메모리에 있는 상태에서 실행하세요)

def run_budget_fixed_battle_500(budget=500):
    dates = sorted(daily_acc['date'].unique())
    report = []
    
    for d in dates:
        day_df = daily_acc[daily_acc['date'] == d].copy()
        
        # 해당 날짜의 전체 데이터가 예산보다 적으면 스킵
        if len(day_df) < budget: continue
        
        # 각 모델별 정렬
        v2_sorted = day_df.sort_values('v2_prob', ascending=False)
        v3_sorted = day_df.sort_values('v3_prob', ascending=False)
        max_sorted = day_df.sort_values('max_score', ascending=False)
        
        # --- [1] 단독 모델 (각 500명) ---
        v2_hits = int(v2_sorted.head(budget)['is_laundering'].sum())
        v3_hits = int(v3_sorted.head(budget)['is_laundering'].sum())
        max_hits = int(max_sorted.head(budget)['is_laundering'].sum())
        
        # --- [2] 50:50 번갈아 채우기 (Union 500명 고정) ---
        union_set = set()
        v2_ids = v2_sorted['account_id'].values
        v3_ids = v3_sorted['account_id'].values
        v2_idx, v3_idx = 0, 0
        
        while len(union_set) < budget:
            if v2_idx < len(v2_ids):
                union_set.add(v2_ids[v2_idx])
                v2_idx += 1
            if len(union_set) >= budget: break
            if v3_idx < len(v3_ids):
                union_set.add(v3_ids[v3_idx])
                v3_idx += 1
        
        union_hits = int(day_df[day_df['account_id'].isin(union_set)]['is_laundering'].sum())
        
        report.append({
            "Date": str(d),
            "V2_Top500": v2_hits,
            "V3_Top500": v3_hits,
            "MaxScore_500": max_hits,
            "BudgetFixed_Union_500": union_hits, # 500명 꽉 채운 앙상블
            "Investigated_N": len(union_set)
        })
        
    print("\n" + "="*95)
    print(f"🏆 [공정 배틀] 모든 기법 조사 인원을 {budget}명으로 고정했을 때 검거 수")
    print("="*95)
    print(pd.DataFrame(report).to_pandas() if hasattr(pd.DataFrame(report), 'to_pandas') else pd.DataFrame(report).to_string(index=False))

run_budget_fixed_battle_500(500)


🏆 [공정 배틀] 모든 기법 조사 인원을 500명으로 고정했을 때 검거 수
      Date  V2_Top500  V3_Top500  MaxScore_500  BudgetFixed_Union_500  Investigated_N
2022-09-14        477        428           478                    474             500
2022-09-15        482        434           484                    478             500
2022-09-16        426        371           423                    407             500
2022-09-17        500        500           500                    500             500
2022-09-18        500        500           500                    500             500
2022-09-19        500        500           500                    500             500
2022-09-20        500        500           500                    500             500
2022-09-21        500        500           500                    500             500
